In [1]:
!pip install -q datasets transformers[sentencepiece] rouge-score accelerate huggingface_hub

  Preparing metadata (setup.py) ... done


In [2]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
import torch
import numpy as np

print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

# Load dataset
dataset = load_dataset("harouzie/vietnews")

# Lấy mẫu
train_data = dataset['train'].shuffle(seed=42).select(range(4000))
val_data = dataset['validation'].shuffle(seed=42).select(range(400))

print(f"Train: {len(train_data)} mẫu")
print(f"Val:   {len(val_data)} mẫu")

# Xem thử 1 mẫu
print("\n--- Mẫu số 0 ---")
print("Article (200 ký tự đầu):", train_data[0]['article'][:200])
print("Abstract:", train_data[0]['abstract'])

GPU available: True
GPU name: Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/575 [00:00<?, ?B/s]

data/train-00000-of-00001-faa04287fdc0e2(…):   0%|          | 0.00/170M [00:00<?, ?B/s]

data/validation-00000-of-00001-10a6e2d4a(…):   0%|          | 0.00/38.3M [00:00<?, ?B/s]

data/test-00000-of-00001-d0d073af901f3b4(…):   0%|          | 0.00/38.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/99134 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/22184 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22498 [00:00<?, ? examples/s]

Train: 4000 mẫu
Val:   400 mẫu

--- Mẫu số 0 ---
Article (200 ký tự đầu): Hầu_hết dự_án BT đầu_tư bằng hình_thức chỉ_định thầu , không qua đấu_giá . Đến khi giao đất cho nhà_đầu_tư , địa_phương cũng không đấu_giá quyền sử_dụng đất . Đất lại được giao ngay sau khi phê_duyệt 
Abstract: Hàng_loạt những bất_cập , hạn_chế trong hình_thức đầu_tư BT đã được nêu ra tại hội_thảo " Cơ_chế đầu_tư BT - Những vấn_đề đặt ra và giải_pháp hoàn_thiện " , do Kiểm_toán Nhà_nước tổ_chức hôm 19-10 .


In [4]:
MODEL_NAME = "vinai/bartpho-syllable"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    # Tokenize article (input)
    inputs = tokenizer(
        examples['article'],
        max_length=512,
        truncation=True,
        padding='max_length'
    )
    # Tokenize abstract (label) - BARTpho dùng cùng tokenizer, không cần as_target_tokenizer
    labels = tokenizer(
        examples['abstract'],
        max_length=128,
        truncation=True,
        padding='max_length'
    )
    inputs['labels'] = labels['input_ids']
    return inputs

print("Đang tokenize...")
tokenized_train = train_data.map(
    preprocess_function,
    batched=True,
    remove_columns=train_data.column_names
)
tokenized_val = val_data.map(
    preprocess_function,
    batched=True,
    remove_columns=val_data.column_names
)
print("Tokenize xong!")

Đang tokenize...


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenize xong!


In [7]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./visum-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=True,
    logging_steps=100,
    save_total_limit=2,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,       # ← ĐÃ SỬA: tokenizer → processing_class
    data_collator=data_collator,
)

print("BẮT ĐẦU TRAIN...")
print("Dự kiến 5 epochs × ~15 phút = ~75 phút")

trainer.train()
print("TRAIN XONG!")

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


BẮT ĐẦU TRAIN...
Dự kiến 5 epochs × ~15 phút = ~75 phút


Epoch,Training Loss,Validation Loss
1,0.809844,0.722767
2,0.699212,0.697150
3,0.615546,0.694343
4,0.547030,0.698441
5,0.493265,0.702104


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TRAIN XONG!


In [8]:
# Lưu model đã train
trainer.save_model("./visum-final")
tokenizer.save_pretrained("./visum-final")
print("✅ Đã lưu model vào ./visum-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Đã lưu model vào ./visum-final


In [19]:
from huggingface_hub import notebook_login
notebook_login()

In [21]:
from huggingface_hub import HfApi
api = HfApi()

# Dùng OrdinaryAI (tên người dùng thật) thay vì Ordinary-AI-Engineer
api.create_repo(repo_id="OrdinaryAI/visum-model", exist_ok=True)
api.upload_folder(
    folder_path="./visum-final",
    repo_id="OrdinaryAI/visum-model",
    repo_type="model",
)

print("✅ Done!")
print("🔗 https://huggingface.co/OrdinaryAI/visum-model")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...l/sentencepiece.bpe.model: 100%|##########| 5.07MB / 5.07MB            

  ...m-final/model.safetensors:   0%|          |  131kB / 1.91GB            

  ...m-final/training_args.bin:  89%|########9 | 4.75kB / 5.33kB            

✅ Done!
🔗 https://huggingface.co/OrdinaryAI/visum-model
